In [1]:
#Paqueting
import pandas as pd
import re
import spacy
import nltk
import csv
import gensim
from collections import Counter
from nltk.corpus import wordnet as wn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
!python -m spacy download es_core_news_sm #!!!No borrar, cargar siempre antes de la celda que sigue

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 24.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


###Limpieza y carga de archivo

In [3]:
#Cosas en español
nltk.download('wordnet')
nltk.download('omw')
nlp = spacy.load("es_core_news_sm")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw to /root/nltk_data...


In [4]:
#Conectores discursivos a eliminar
con_disc = []
with open("/content/drive/MyDrive/conectores dfiscursivos.csv", newline='') as file:
    csv_reader = csv.reader(file)
    for f in csv_reader:
        con_disc.extend(f)

In [5]:
def limpiar(texto):
    if pd.isna(texto):
        return ""
    texto = texto.lower()
    texto = re.sub(r"\d+", "", texto)
    texto = re.sub(r"[^\w\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    doc = nlp(texto)
    tokens = [
        token.lemma_ for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token.lemma_) > 2
        and token.pos_ != "VERB"
        and token.lemma_ not in con_disc
    ]

    return " ".join(tokens)

In [6]:
#Para quitar las filas similares/iguales
def quitar(df, threshold=0.95):
    df['limpiando'] = df['DESC_CAUSA_PROCESADO']
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df['limpiando'])
    cosine_similarities = cosine_similarity(tfidf_matrix)
    rows_to_remove = set()
    for i in range(len(cosine_similarities)):
        for j in range(i + 1, len(cosine_similarities)):
            if cosine_similarities[i, j] > threshold:
                rows_to_remove.add(j)
    filtered_df = df[~df.index.isin(rows_to_remove)]
    return filtered_df.drop(columns=['limpiando'])



In [7]:
path = '/content/drive/MyDrive/c19-23-1011.xlsx'
dfo = pd.read_excel(path)

In [8]:
df = dfo.copy() #!!!Para crear y reiniciar los datos

In [9]:
df[df['DESC_CAUSA_CP'] != ""] #quitar las vacías

,CICLO,RAMO,KA_META_TIPO_JUST_DOCUMENTO,PP,IDEN_PROY,DESC_CAUSA_CP
0,2019,2,11,P 003,P,Se atienden de manera pronta y expedita las so...
1,2019,2,11,P 002,P,El ejercicio oportuno y racional del gasto por...
2,2019,2,11,M 001,M,Se cumple con la meta establecida ejerciendo l...
3,2019,2,9,O 001,O,El objetivo de inhibición depende de la activi...
4,2019,2,9,O 001,O,El pasado 9 de diciembre de 2019 se publicó en...
...,...,...,...,...,...,...
28896,2023,53,11,R 584,R,La EPS CFE Generación V obtuvo el 100% del ind...
28897,2023,53,9,E 561,E,La Comisión Federal de Electricidad a través d...
28898,2023,53,10,E 579,E,Se presentaron mayor número de eventos en el R...
28899,2023,53,10,E 562,E,La Comisión Federal de Electricidad (CFE) a tr...


In [10]:
df["DESC_CAUSA_PROCESADO"] = df['DESC_CAUSA_CP'].apply(limpiar)

In [ ]:
df = quitar(df)

In [ ]:
#Obtener las stop words de las DESC_CAUSAS
pbr_contadas = Counter(" ".join(df["DESC_CAUSA_PROCESADO"]).split())
stop_propias = {word for word, count in pbr_contadas.items() if count > 0.075 * len(df)}
print(stop_propias)

###Aprender

In [ ]:
#Otras stop words
otras = ["año", "enero", "febrero", "marzo", "abril", "mayo", "junio", "agosto", "septiembre", "octubre", "noviembre", "diciembre", "programación", "ciclo", "méxico", "anual", "mensual", "ramo", "generado", "realizado", "registrado","presupuestal", "presupuesto", "presupuestario", "vez", "metas", "indicadores", "realizar"]

In [ ]:
#Descargar las stopwords disponibles en español
nltk.download('stopwords')
stopword_es = nltk.corpus.stopwords.words('spanish')

In [ ]:
#Crear una lista de stopwords con las de la librería y las obtenidas
stop_propias = list(stop_propias)
stopwords = stopword_es + stop_propias + otras

In [ ]:
#Limpiar ooooooooooooooootra vez con las stopwords que encontró en la primera limpieza :D
def limpiar(texto):
    if pd.isna(texto):
        return ""
    texto = ' '.join([palabra for palabra in texto.split() if palabra not in stopwords])
    return texto


In [ ]:
df["DESC_CAUSA_PROCESADO"] = df['DESC_CAUSA_PROCESADO'].apply(limpiar)

In [ ]:
#Modelosss para ver cuál es mejor
X = df['DESC_CAUSA_PROCESADO']
y = df['KA_META_TIPO_JUST_DOCUMENTO']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2000)

vectorizer = TfidfVectorizer(stop_words=stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = RandomForestClassifier(class_weight='balanced',n_estimators=150, random_state=2000)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)

print(classification_report(y_test, y_pred,zero_division=1))

df_grouped = df.groupby('KA_META_TIPO_JUST_DOCUMENTO')['DESC_CAUSA_PROCESADO'].apply(' '.join).reset_index()
X_grouped = vectorizer.transform(df_grouped['DESC_CAUSA_PROCESADO'])


terms = vectorizer.get_feature_names_out()

for idx, row in df_grouped.iterrows():
    tfidf_scores = X_grouped[idx, :].toarray().flatten()
    word_score_pairs = [(terms[i], tfidf_scores[i]) for i in range(len(terms))]
    sorted_word_score_pairs = sorted(word_score_pairs, key=lambda x: x[1], reverse=True)
    top_keywords = [pair[0] for pair in sorted_word_score_pairs[:10]]

    print(f"KA_META: {df_grouped.iloc[idx]['KA_META_TIPO_JUST_DOCUMENTO']}")
    print(f"Palabras: {', '.join(top_keywords)}\n")

###KA_META 9

####Topic modelling 2.0

In [ ]:
df_9_1 = df[df['KA_META_TIPO_JUST_DOCUMENTO'] == 9]

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words=stopwords)
X = tfidf_vectorizer.fit_transform(df_9_1['DESC_CAUSA_PROCESADO'])

lda = LatentDirichletAllocation(n_components=11, random_state=2000)
lda.fit(X)

terms = tfidf_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    print(f"Tema {topic_idx}:")
    print([terms[i] for i in topic.argsort()[:-10 - 1:-1]])

Tema 0:
['carpetas', 'obras', 'contó', 'cartera', 'juicio', 'distritos', 'oral', 'reprogramación', 'delitos', 'obedeció']
Tema 1:
['centros', 'previsión', 'laboral', 'sarscov', 'sanitaria', 'jóvenes', 'disminución', 'virus', 'pesos', 'mujeres']
Tema 2:
['inbal', 'estudiantes', 'artes', 'bellas', 'literatura', 'escuela', 'educación', 'sitios', 'mwh', 'antropología']
Tema 3:
['asuntos', 'excedentes', 'modalidad', 'opera', 'individuales', 'promedio', 'adelanto', 'costo', 'conciliación', 'mujeres']
Tema 4:
['alumnos', 'educación', 'control', 'becas', 'estudiantes', 'unidades', 'docentes', 'superior', 'escolar', 'estudios']
Tema 5:
['asignaron', 'presupuestales', 'inferior', 'restricciones', 'superficie', 'multidimensional', 'energía', 'bienes', 'tiendas', 'supervisiones']
Tema 6:
['averiguaciones', 'previas', 'etiquetó', 'tratamiento', 'pef', 'puntos', 'comportamiento', 'obedeció', 'cursos', 'porcentuales']
Tema 7:
['cuenta', 'expedientes', 'posible', 'apoyos', 'reporta', 'resultados', 'co

### Similitud

In [ ]:
#Modificar y aplicar las palabras con el mejor resultado

In [ ]:
KA_DESC = ["Programación original deficiente", "Emergencias provocadas por accidentes y/o fenómenos naturales adversos", "Menor demanda de bienes y servicios", "Retrasos en los trámites para el ejercicio presupuestario por parte de la Unidad Responsable (UR)", "Incumplimiento o retraso en los trámites para el ejercicio presupuestario por parte de instancias gubernamentales diferentes a la UR.","Incumplimiento o inconformidades de proveedores y contratistas, así como por oposición de grupos sociales", "Modificación de atribuciones institucionales por disposiciones normativas", "Incumplimiento por situaciones normativas extrapresupuestarias ajenas a la UR", "Otras causas que por su naturaleza no es posible agrupar", "Otras explicaciones a las variaciones, cuando se trate de resultados por encima del 100 por ciento de cumplimiento", "La meta del indicador del desempeño fue cumplida", "Emergencias sanitarias"]
df_grouped = df.groupby('KA_META_TIPO_JUST_DOCUMENTO')['DESC_CAUSA_PROCESADO'].apply(' '.join).reset_index()
texto_completo = KA_DESC + df_grouped['DESC_CAUSA_PROCESADO'].tolist()
X = vectorizer.fit_transform(texto_completo)
resultados_similitud = []
for idx, causa in enumerate(df_grouped['DESC_CAUSA_PROCESADO']):
    ka_idx = df_grouped['KA_META_TIPO_JUST_DOCUMENTO'][idx] - 1
    ka_desc = KA_DESC[ka_idx]
    causa_vector = vectorizer.transform([causa])
    ka_desc_vector = vectorizer.transform([ka_desc])
    distance = euclidean_distances(causa_vector, ka_desc_vector)[0][0] #Euclidiana
    similitud = 1 / (1 + distance)
    resultados_similitud.append({
        "KA_META_TIPO_JUST_DOCUMENTO": df_grouped['KA_META_TIPO_JUST_DOCUMENTO'][idx],
        "DESC_CAUSA_PROCESADO": causa,
        "KA_DESC": ka_desc,
        "Similitud": similitud
    })


In [ ]:
df_resultados = pd.DataFrame(resultados_similitud)
df_resultados.to_excel("/content/drive/MyDrive/similitudt.xlsx")

### SImilitud por cada renglón

In [ ]:
resultados_similitud = []
for idx, row in df.iterrows():
    causa = row['DESC_CAUSA_PROCESADO']
    ka_meta = row['KA_META_TIPO_JUST_DOCUMENTO']
    ka_desc = KA_DESC[ka_meta - 1]
    causa_vector = vectorizer.transform([causa])
    ka_desc_vector = vectorizer.transform([ka_desc])
    distance = euclidean_distances(causa_vector, ka_desc_vector)[0][0]
    similitud = 1 / (1 + distance)
    resultados_similitud.append({
        "KA_META_TIPO_JUST_DOCUMENTO": ka_meta,
        "DESC_CAUSA_PROCESADO": causa,
        "KA_DESC": ka_desc,
        "Similitud": similitud
    })

In [ ]:
df_similitudes = pd.DataFrame(resultados_similitud)
df_similitudes

,KA_META_TIPO_JUST_DOCUMENTO,DESC_CAUSA_PROCESADO,KA_DESC,Similitud
0,11,atienden manera pronta expedita ciudadanas cua...,La meta del indicador del desempeño fue cumplida,0.414214
1,11,oportuno racional gasto unidades responsables ...,La meta del indicador del desempeño fue cumplida,0.414214
2,11,cumple establecida ejerciendo totalidad destin...,La meta del indicador del desempeño fue cumplida,0.414214
3,9,objetivo inhibición depende genere responsabil...,Otras causas que por su naturaleza no es posib...,0.414214
4,9,pasado publicó diario oficial federación nuevo...,Otras causas que por su naturaleza no es posib...,0.414214
...,...,...,...,...
23628,9,comisión electricidad eps generación iv establ...,Otras causas que por su naturaleza no es posib...,0.414214
23629,11,eps cfe generación v obtuvo enviar ofertas ven...,La meta del indicador del desempeño fue cumplida,0.414214
23630,9,comisión electricidad eps generación vi establ...,Otras causas que por su naturaleza no es posib...,0.414214
23631,10,presentaron eventos red transmisión rnt afecta...,"Otras explicaciones a las variaciones, cuando ...",0.414214


In [ ]:
df_similitudes.to_excel("/content/drive/MyDrive/output1.xlsx")